# Bharat-Tiny-LLM v3.5 — Mixed Math+Chat SFT (Kaggle T4)

Two modes controlled by `RUN_MODE`:
- **`smoke`** — 500 rows, 1 epoch, ~5 min → validates pipeline end-to-end (loss decreasing, generation non-garbage)
- **`full`** — all ~44K gold rows, 1–2 epochs → product candidate

Data: `eulogik/bharat-v3-mixed-sft-v34` (43,887 gold-standard rows, Yojana garbage removed).

| Stage | Rows | Epochs | Est. 1×T4 | Est. 2×T4 DDP |
|---|---:|---:|---:|---:|
| smoke | 500 | 1 | ~5 min | — |
| full | 43,887 | 1 | ~2.2 hr | ~1.1 hr |
| full | 43,887 | 2 | ~4.3 hr | ~2.2 hr |

Base: already-merged v3 (`eulogik/Bharat-Tiny-LLM-v3`) + fresh LoRA r=32.


In [ ]:
#@title 0 · CONFIG {display-mode:"form"}
import os, json, time
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

RUN_MODE = "smoke"  #@param ["smoke", "full"]
EPOCHS = 1  #@param {type:"slider", min:1, max:3, step:1}
LORA_R = 32
LORA_ALPHA = 64
LR = 1e-4 if RUN_MODE == "smoke" else 2e-5
PER_DEVICE_BS = 2
GRAD_ACCUM = 4
SEQ_LEN = 1024
MATH_UPSAMPLE = 3  # duplicate GSM-style (####) rows to rebalance math vs chat
TARGET_REPO = "eulogik/Bharat-Tiny-LLM-v3.5"
MAX_SESSION_HOURS = 7.5   # leave buffer in 8-hr quota

WORK = Path("/kaggle/working/brahmi_v35")
WORK.mkdir(parents=True, exist_ok=True)

BASE_MODEL = "eulogik/Bharat-Tiny-LLM-v3"
MIXED_DATA = "eulogik/bharat-v3-mixed-sft-v34"
OUT_DIR = WORK / "sft_out"
HEARTBEAT = WORK / "progress.json"

# HF token (retries + public fallback — model/dataset are public)
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN") or ""
if not HF_TOKEN:
    for attempt in range(3):
        try:
            from kaggle_secrets import UserSecretsClient
            c = UserSecretsClient()
            for name in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HF_TOKEN_V34", "huggingface"):
                try:
                    v = c.get_secret(name)
                    if v:
                        HF_TOKEN = v
                        print(f"loaded HF token from secret: {name}")
                        break
                except Exception:
                    pass
            if HF_TOKEN:
                break
        except Exception as e:
            print(f"secrets attempt {attempt+1}: {e}")
            time.sleep(2)
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import whoami
        print("HF auth:", whoami(token=HF_TOKEN)["name"])
    except Exception as e:
        print("whoami failed (continuing):", e)
else:
    print("HF_TOKEN not available — using anonymous access (model+dataset are public)")

STATE_PATH = WORK / "state.json"
STATE = json.loads(STATE_PATH.read_text()) if STATE_PATH.exists() else {}

def save_state():
    STATE_PATH.write_text(json.dumps(STATE))

print(f"mode={RUN_MODE} epochs={EPOCHS} lr={LR} token={'set' if HF_TOKEN else 'anonymous'}")


In [ ]:
#@title 1 · Setup deps {display-mode:"form"}
import subprocess, sys
# Match proven working stack from brahmi_v3_kaggle.ipynb
pkgs = [
    "transformers>=4.53,<5",
    "peft>=0.15,<0.20",
    "accelerate>=1.0",
    "datasets>=3.0",
    "safetensors",
    "huggingface_hub",
    "sentencepiece",
]
subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", *pkgs])
# Kaggle ships torchao that clashes with peft
subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import transformers
print("transformers", transformers.__version__)
import peft
print("peft", peft.__version__)
import torch
print(f"torch {torch.__version__} cuda={torch.cuda.is_available()} gpus={torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    mem = getattr(p, "total_memory", None) or getattr(p, "total_mem", 0)
    print(f"  GPU{i}: {p.name} {mem/1e9:.1f}GB")

# Install tokenizer workaround AFTER transformers is the pinned version
from transformers.tokenization_utils_base import PreTrainedTokenizerBase
if not getattr(PreTrainedTokenizerBase, "_v34_init_wrapped", False):
    _orig_init = PreTrainedTokenizerBase.__init__
    def _v34_init(self, *args, **kwargs):
        est = kwargs.get("extra_special_tokens")
        if isinstance(est, (list, tuple, set)):
            kwargs["extra_special_tokens"] = {}
        return _orig_init(self, *args, **kwargs)
    PreTrainedTokenizerBase.__init__ = _v34_init
    PreTrainedTokenizerBase._v34_init_wrapped = True
    print("tokenizer extra_special_tokens workaround installed")


In [ ]:
#@title 2 · Load base v3 + LoRA {display-mode:"form"}
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

if not STATE.get("model_loaded"):
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN or None, trust_remote_code=True)
    if tok.pad_token is None and tok.eos_token is not None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        dtype=torch.float16,
        attn_implementation="sdpa",
        token=HF_TOKEN or None,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    model = model.to("cuda:0")
    for p in model.parameters():
        p.requires_grad = False

    lcfg = LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        bias="none", task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lcfg)
    model.enable_input_require_grads()
    model.config.use_cache = False
    torch.cuda.empty_cache()
    model.print_trainable_parameters()
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"trainable: {n_params/1e6:.1f}M params | load {time.time()-t0:.1f}s")
    print(f"world_size={torch.cuda.device_count()} (Trainer DataParallel if >1)")

    STATE["model_loaded"] = True
    save_state()
else:
    print("model already loaded")


In [ ]:
#@title 3 · Load mixed dataset + tokenize {display-mode:"form"}
from datasets import load_dataset

if not STATE.get("tok_done"):
    t0 = time.time()
    ds = load_dataset(MIXED_DATA, split="train", token=HF_TOKEN)
    print(f"raw rows: {len(ds)}")

    if RUN_MODE == "smoke":
        ds = ds.shuffle(seed=42).select(range(min(500, len(ds))))
        print(f"smoke subset: {len(ds)}")
    elif RUN_MODE == "full" and len(ds) > 44000:
        # safety cap
        ds = ds.shuffle(seed=42).select(range(44000))

    # hold out 200 for eval
    split = ds.train_test_split(test_size=(50 if RUN_MODE == 'smoke' else 200), seed=42)
    train_ds, eval_ds = split["train"], split["test"]
    print(f"train={len(train_ds)} eval={len(eval_ds)}")

    # v3.5 fix: upsample math rows (GSM #### answers) to counter chat dilution
    # v3 56% -> v3.3 24% -> v3.4 ~13% GSM trend tracked to math dilution (~17% math)
    if MATH_UPSAMPLE > 1:
        def _is_math(ex):
            try:
                msgs = ex["messages"]
                if isinstance(msgs, str):
                    msgs = json.loads(msgs)
                if isinstance(msgs, dict) and "messages" in msgs:
                    msgs = msgs["messages"]
                a = msgs[-1].get("content", "") if isinstance(msgs, list) and msgs else ""
                return "####" in str(a)
            except Exception:
                return False
        math_idx = [i for i, ex in enumerate(train_ds) if _is_math(ex)]
        print(f"math rows in train: {len(math_idx)}/{len(train_ds)}")
        if math_idx:
            extra = []
            for _ in range(MATH_UPSAMPLE - 1):
                extra.extend(math_idx)
            from datasets import concatenate_datasets
            train_ds = concatenate_datasets([train_ds, train_ds.select(extra)]).shuffle(seed=42)
            print(f"after x{MATH_UPSAMPLE} math upsample: train={len(train_ds)}")

    def tokenize(ex):
        msgs = ex["messages"]
        if isinstance(msgs, str):
            msgs = json.loads(msgs)
        if isinstance(msgs, dict) and "messages" in msgs:
            msgs = msgs["messages"]
        full = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        prompt = tok.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True)
        full_ids = tok(full, truncation=True, max_length=SEQ_LEN, add_special_tokens=False)["input_ids"]
        prompt_ids = tok(prompt, truncation=True, max_length=SEQ_LEN-1, add_special_tokens=False)["input_ids"]
        plen = min(len(prompt_ids), len(full_ids))
        labels = [-100] * plen + full_ids[plen:]
        # pad to fixed SEQ_LEN so default_data_collator can stack batches
        pad_id = tok.pad_token_id if tok.pad_token_id is not None else (tok.eos_token_id or 0)
        pad_n = SEQ_LEN - len(full_ids)
        if pad_n > 0:
            full_ids = full_ids + [pad_id] * pad_n
            labels = labels + [-100] * pad_n
            attn = [1] * (SEQ_LEN - pad_n) + [0] * pad_n
        else:
            attn = [1] * SEQ_LEN
        return {"input_ids": full_ids, "attention_mask": attn, "labels": labels}

    train_tok = train_ds.map(tokenize, remove_columns=train_ds.column_names, num_proc=4)
    eval_tok = eval_ds.map(tokenize, remove_columns=eval_ds.column_names, num_proc=4)
    train_tok.save_to_disk(str(WORK / "train_tok"))
    eval_tok.save_to_disk(str(WORK / "eval_tok"))
    print(f"tokenized in {time.time()-t0:.1f}s | sample len={len(train_tok[0]['input_ids'])}")
    STATE["tok_done"] = True
    save_state()
else:
    from datasets import load_from_disk
    train_tok = load_from_disk(str(WORK / "train_tok"))
    eval_tok = load_from_disk(str(WORK / "eval_tok"))
    print(f"loaded from disk train={len(train_tok)} eval={len(eval_tok)}")


In [ ]:
#@title 4 · Train {display-mode:"form"}
from transformers import Trainer, TrainingArguments, TrainerCallback

class Heartbeat(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kw):
        if state.is_world_process_zero and logs and "loss" in logs:
            HEARTBEAT.write_text(json.dumps({
                "step": state.global_step, "loss": logs.get("loss"),
                "eval_loss": logs.get("eval_loss"), "ts": time.time(),
            }))
            print(f"  [step {state.global_step}] loss={logs.get('loss'):.4f}", flush=True)
    def on_evaluate(self, args, state, control, metrics=None, **kw):
        if state.is_world_process_zero and metrics:
            print(f"  [eval step {state.global_step}] eval_loss={metrics.get('eval_loss')}", flush=True)
            try:
                prev = json.loads(HEARTBEAT.read_text()) if HEARTBEAT.exists() else {}
            except Exception:
                prev = {}
            prev.update({"step": state.global_step, "eval_loss": metrics.get("eval_loss"), "ts": time.time()})
            HEARTBEAT.write_text(json.dumps(prev))

if not STATE.get("train_done"):
    total_steps_est = (len(train_tok) * EPOCHS) // (PER_DEVICE_BS * GRAD_ACCUM)
    args = TrainingArguments(
        output_dir=str(OUT_DIR),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=PER_DEVICE_BS,
        per_device_eval_batch_size=PER_DEVICE_BS,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=500 if RUN_MODE == "full" else 50,
        save_strategy="steps",
        save_steps=500 if RUN_MODE == "full" else 50,
        save_total_limit=2,
        bf16=False,
        fp16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        dataloader_num_workers=0,
        dataloader_pin_memory=False,
        report_to=[],
        seed=42,
        remove_unused_columns=False,
    )
    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_tok, eval_dataset=eval_tok,
        callbacks=[Heartbeat()],
    )
    t0 = time.time()
    result = trainer.train()
    elapsed = time.time() - t0
    print(f"DONE in {elapsed/60:.1f} min | final loss={result.training_loss:.4f}")
    (WORK / "train_result.json").write_text(json.dumps({
        "training_loss": result.training_loss,
        "elapsed_min": elapsed/60,
        "train_samples": len(train_tok),
        "mode": RUN_MODE, "epochs": EPOCHS, "lr": LR,
    }, indent=2))
    trainer.save_model(str(OUT_DIR / "final_adapter"))
    tok.save_pretrained(str(OUT_DIR / "final_adapter"))
    STATE["train_done"] = True
    save_state()
else:
    print("training already done")


In [ ]:
#@title 5 · Quick generation sanity check {display-mode:"form"}
from transformers import pipeline as hf_pipeline
import re

adapter_path = str(OUT_DIR / "final_adapter")
print(f"loading adapter from {adapter_path}...")

try:
    del model, trainer
except Exception:
    pass
torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, dtype=torch.float16, device_map={"": 0},
    token=HF_TOKEN or None, trust_remote_code=True,
)
from peft import PeftModel
model_inf = PeftModel.from_pretrained(base, adapter_path)
model_inf.eval()

gen = hf_pipeline("text-generation", model=model_inf, tokenizer=tok)

prompts = [
    "5 सेब में से 2 खा लिए, बचे कितने?",
    "नमस्ते, आपका नाम क्या है?",
    "15 + 27 = ?",
    "आज मौसम कैसा है?",
]
texts = []
for p in prompts:
    messages = [
        {"role": "system", "content": "You are Bharat-Tiny-LLM, a helpful bilingual assistant for Hindi and Hinglish. Answer in the same language the user writes. Be concise."},
        {"role": "user", "content": p},
    ]
    out = gen(messages, max_new_tokens=80, do_sample=False,
              temperature=None, top_p=None, top_k=None,
              pad_token_id=tok.eos_token_id)
    text = out[0]["generated_text"][-1]["content"]
    texts.append(text)
    print(f"\nQ: {p}")
    print(f"A: {text[:300]!r}")

junk_re = re.compile(r"[\u0E00-\u0E7F\u4E00-\u9FFF\u0600-\u06FF]{4,}")
last = texts[-1] if texts else ""
print("\n--- foreign-script junk check:", "FAIL" if junk_re.search(last) else "PASS")
for t in texts:
    if junk_re.search(t):
        print("FOREIGN SCRIPT DETECTED in:", t[:200])
        break
else:
    print("ALL PROMPTS CLEAN")


In [ ]:
#@title 6 · Merge adapter + upload v3.5 {display-mode:"form"}
if STATE.get("train_done") and not STATE.get("export_done"):
    import torch
    from peft import PeftModel
    from huggingface_hub import HfApi

    t0 = time.time()
    print("loading base fp32 for merge...")
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, dtype=torch.float32, token=HF_TOKEN, trust_remote_code=True,
    )
    model_m = PeftModel.from_pretrained(base, str(OUT_DIR / "final_adapter"))
    model_m = model_m.merge_and_unload()
    model_m.half()

    export = WORK / "export_v35"
    model_m.save_pretrained(str(export), safe_serialization=True, max_shard_size="4GB")
    tok.save_pretrained(str(export))
    print(f"merged + saved in {time.time()-t0:.1f}s")

    import os as _os
    api = HfApi(token=HF_TOKEN) if HF_TOKEN else HfApi()
    try:
        api.create_repo(TARGET_REPO, exist_ok=True, repo_type="model")
    except Exception as e:
        print("create_repo:", e)
    # per-file upload (upload_folder hangs on big safetensors)
    try:
        import time as _t
        files = sorted(_os.listdir(str(export)))
        files = sorted(files, key=lambda n: (n != "README.md", n))
        for name in files:
            p = _os.path.join(str(export), name)
            t1 = _t.time()
            print(f"  uploading {name} ({_os.path.getsize(p)} bytes)...", flush=True)
            api.upload_file(path_or_fileobj=p, path_in_repo=name, repo_id=TARGET_REPO, repo_type="model", commit_message=f"Add {name}")
            print(f"    done in {_t.time()-t1:.1f}s", flush=True)
        print(f"uploaded {TARGET_REPO}")
        STATE["export_done"] = True
    except Exception as e:
        print(f"HF upload failed (token={'set' if HF_TOKEN else 'missing'}): {e}")
        print(f"merged weights left at {export} — download via kaggle kernels output")
        STATE["export_saved_local"] = True
    save_state()
else:
    print(f"skip export (train_done={STATE.get('train_done')}, export_done={STATE.get('export_done')})")


In [ ]:
#@title 7 · Status {display-mode:"form"}
print("STATE:", STATE)
if HEARTBEAT.exists():
    print("heartbeat:", HEARTBEAT.read_text())
if (WORK / "train_result.json").exists():
    print("result:", (WORK / "train_result.json").read_text())
if (WORK / "sft_out").exists():
    import os
    for root, dirs, files in os.walk(WORK / "sft_out"):
        for f in files:
            print(os.path.join(root, f))
